# 04.3 Checkpoint and Logging

The goal of this notebook is to make training results recoverable and reproducible. A checkpoint lets you restore model or training state. Logs and config files explain what happened during the run and which settings produced the result.

Without these artifacts, a good model can become hard to reproduce and a bad model can become hard to diagnose.

## Learning Goals

After this notebook, you should be able to:

1. Understand why saving only the model is often not enough.
2. Save `model_state
3. Log loss and accuracy during training.
4. Load a checkpoint and restore a model.
5. Understand the role of random seeds in reproducibility.

In [ ]:
import csv
import json
import random
import tempfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(42)

## Prepare a Minimal Experiment

We again use a small binary classification toy task; the focus is not the task itself, but the logging and recovery workflow.


In [ ]:
config = {
    "seed": 42,
    "batch_size": 32,
    "hidden_dim": 16,
    "lr": 0.1,
    "epochs": 8,
}

set_seed(config["seed"])
workdir = Path(tempfile.mkdtemp(prefix="phase4_checkpoint_logging_"))
best_ckpt_path = workdir / "best_checkpoint.pt"
last_ckpt_path = workdir / "last_checkpoint.pt"
history_json_path = workdir / "history.json"
history_csv_path = workdir / "history.csv"
config_json_path = workdir / "config.json"

x = torch.randn(320, 2)
y = (x[:, 0] - 0.5 * x[:, 1] > 0).long()

train_ds = TensorDataset(x[:256], y[:256])
val_ds = TensorDataset(x[256:], y[256:])
train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True)
val_loader = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False)

print("workdir =", workdir)
print("train size =", len(train_ds))
print("val size =", len(val_ds))

In [ ]:
class TinyClassifier(nn.Module):
    def __init__(self, in_dim=2, hidden_dim=16, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)


def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    total_correct = 0
    total_items = 0

    for xb, yb in loader:
        with torch.set_grad_enabled(is_train):
            logits = model(xb)
            loss = loss_fn(logits, yb)

        if is_train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * xb.size(0)
        total_correct += (preds == yb).sum().item()
        total_items += xb.size(0)

    return total_loss / total_items, total_correct / total_items

## Train and Log History

A practical training history records the epoch number, training loss, training accuracy, validation loss, and validation accuracy. Those values let you inspect whether the model is learning, whether validation performance is improving, and whether overfitting begins after a certain epoch.

The history is useful even if you never load a checkpoint, because it explains the training trajectory rather than only the final result.

In [ ]:
model = TinyClassifier(hidden_dim=config["hidden_dim"])
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=config["lr"])

history = []
best_val_loss = float("inf")
best_epoch = None

for epoch in range(1, config["epochs"] + 1):
    train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)

    row = {
        "epoch": epoch,
        "train_loss": round(train_loss, 6),
        "train_acc": round(train_acc, 6),
        "val_loss": round(val_loss, 6),
        "val_acc": round(val_acc, 6),
    }
    history.append(row)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "config": config,
                "history": history,
                "best_val_loss": best_val_loss,
            },
            best_ckpt_path,
        )

    print(
        f"epoch={epoch:02d} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )

torch.save(
    {
        "epoch": config["epochs"],
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "config": config,
        "history": history,
    },
    last_ckpt_path,
)

print("best_epoch =", best_epoch)
print("best_ckpt_path =", best_ckpt_path)
print("last_ckpt_path =", last_ckpt_path)

## Save Config and History as Files

Saving config and history as text files makes experiments easier to inspect. You can read the settings and metric trajectory without loading a PyTorch checkpoint. These files are also convenient for plotting, reporting, and comparing runs later.

In [ ]:
config_json_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
history_json_path.write_text(json.dumps(history, indent=2), encoding="utf-8")

with history_csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=history[0].keys())
    writer.writeheader()
    writer.writerows(history)

print("saved files:")
for path in [config_json_path, history_json_path, history_csv_path, best_ckpt_path, last_ckpt_path]:
    print("-", path.name, "exists=", path.exists())

## Load a Checkpoint and Restore the Model

Here we use the `last checkpoint` to verify that restoration works correctly.


In [ ]:
loaded = torch.load(last_ckpt_path, map_location="cpu")
restored_model = TinyClassifier(hidden_dim=loaded["config"]["hidden_dim"])
restored_model.load_state_dict(loaded["model_state"])
restored_model.eval()
model.eval()

sample_x = x[256:261]
original_logits = model(sample_x)
restored_logits = restored_model(sample_x)
original_preds = original_logits.argmax(dim=1)
restored_preds = restored_logits.argmax(dim=1)

print("loaded epoch =", loaded["epoch"])
print("loaded config =", loaded["config"])
print("original_preds =", original_preds)
print("restored_preds =", restored_preds)
print("predictions identical / predictions identical =", torch.equal(original_preds, restored_preds))

In [ ]:
best_loaded = torch.load(best_ckpt_path, map_location="cpu")
print("best checkpoint epoch =", best_loaded["epoch"])
print("best checkpoint val loss =", best_loaded["best_val_loss"])
print("history length inside checkpoint =", len(best_loaded["history"]))

## A Practical Habit

For each experiment, leave behind enough artifacts to understand and reproduce the run. A config file records the settings. A best checkpoint preserves the model with the best validation performance. A last checkpoint preserves the final training state. Together, they make the run easier to resume, compare, and explain.

In [ ]:
# Exercise 1
#
# Explain in full sentences:
# Why is saving only model.state_dict() often not enough?
#
# Your answer should distinguish inference from resuming training and mention
# optimizer state, epoch/step, config, or metric history.

Exercise 1 Reference Answer

Because you may also need to restore the optimizer state, epoch, experiment config, and metric history.

These pieces of information are important both for resuming training and reproducing experiments.


In [ ]:
# Exercise 2
#
# Explain the difference between a best checkpoint and a last checkpoint.
#
# Your answer should mention:
# - which one is chosen by validation performance
# - which one represents the final training state
# - why a project may save both

Exercise 2 Reference Answer

- the checkpoint from the epoch with the best validation performance
- the checkpoint saved at the final training step or epoch

Many projects save both, because they serve different purposes.


## Summary

The most important takeaways from this notebook are:

1. a checkpoint is not only model weights; it should include what you need to resume training
2. `config` and `history` should preferably also be saved as separate files
3. both `best checkpoint` and `last checkpoint` are often worth keeping
4. without logs and configs, many experiments are effectively not reproducible